# 🏦 Agent Orchestration Patterns -- Banking Fraud Resolution (Azure AI Foundry Agents)

This notebook compares **5 orchestration patterns** against the same banking fraud scenario, but uses **persistent Azure AI Foundry agents** created through `azure.ai.agents.AgentsClient` instead of ephemeral in-memory MAF agents.

> *A customer reports unauthorized transactions on their business checking account totaling $12,500 over the past 48 hours. They need fraud investigation, account security, and provisional credit.*

| Pattern | How It Works | Best For |
|---------|-------------|----------|
| **Sequential** | Agents run one after another, each building on previous output | Multi-step processes with dependencies |
| **Handoff** | Router analyzes request, delegates to a specialist agent | Dynamic routing to domain experts |
| **Concurrent** | Multiple agents analyze in parallel, results are merged | Independent analyses needed fast |
| **Group Chat** | Agents collaborate in rounds with shared context | Complex decisions requiring debate |
| **Magentic** | Planner decomposes into task ledger, researcher + writer + validator iterate | Complex cases needing structured decomposition and quality gates |


## 1. Setup — Load environment and create `AgentsClient`


In [ ]:
import os
import time
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path

from dotenv import load_dotenv
from azure.identity import DefaultAzureCredential
from azure.ai.agents import AgentsClient
from azure.ai.agents.models import AgentThreadCreationOptions, ThreadMessageOptions, RunStatus

load_dotenv(Path('.env'), override=True)

PROJECT_ENDPOINT = os.getenv('AZURE_AI_ENDPOINT') or os.getenv('AZURE_AI_PROJECT_ENDPOINT')
MODEL_DEPLOYMENT = os.getenv('AZURE_OPENAI_CHAT_DEPLOYMENT_NAME')

if not PROJECT_ENDPOINT:
    raise ValueError('Set AZURE_AI_ENDPOINT (AI Foundry project endpoint) in .env')
if not MODEL_DEPLOYMENT:
    raise ValueError('AZURE_OPENAI_CHAT_DEPLOYMENT_NAME is required in .env')

credential = DefaultAzureCredential()
client = AgentsClient(endpoint=PROJECT_ENDPOINT, credential=credential)

BANKING_TASK = (
    'A customer reports unauthorized transactions on their business checking account '    'totaling $12,500 over the past 48 hours. Three transactions were flagged: '    '$4,200 wire transfer to an unknown account, $5,800 online purchase from an '    'overseas merchant, and $2,500 ATM withdrawal in a different state. '    'The customer needs immediate resolution including fraud investigation, '    'account security measures, and provisional credit assessment.'
)

results = {}

print(f'Project endpoint: {PROJECT_ENDPOINT}')
print(f'Model deployment: {MODEL_DEPLOYMENT}')
print('AgentsClient ready.')


## 2. Helper — Idempotent agent creation


In [ ]:
def get_or_create_agent(client, name, instructions, model, metadata=None):
    metadata = metadata or {}

    for existing in client.list_agents():
        if getattr(existing, 'name', None) == name:
            print(f'Reusing existing agent: {name} ({existing.id})')
            return existing

    created = client.create_agent(
        model=model,
        name=name,
        instructions=instructions,
        metadata=metadata,
    )
    print(f'Created new agent: {name} ({created.id})')
    return created


## 3. Agent definitions


In [ ]:
AGENT_CONFIGS = {
    'sequential_fraud_analyst': {
        'name': 'PatternComparison-Sequential-FraudAnalyst',
        'instructions': 'You are a bank fraud analyst. Analyze the reported transactions for fraud indicators, classify risk, identify suspicious patterns, and recommend investigation priorities.',
        'metadata': {'source': 'pattern_comparison', 'pattern': 'sequential'},
    },
    'sequential_security_agent': {
        'name': 'PatternComparison-Sequential-SecurityAgent',
        'instructions': 'You are a banking account security specialist. Recommend immediate containment actions such as account holds, credential resets, card controls, fraud alerts, and monitoring steps ordered by urgency.',
        'metadata': {'source': 'pattern_comparison', 'pattern': 'sequential'},
    },
    'sequential_credits_agent': {
        'name': 'PatternComparison-Sequential-CreditsAgent',
        'instructions': 'You are a provisional credit and claims specialist. Assess provisional credit eligibility, required documentation, verification checkpoints, and customer communication needs.',
        'metadata': {'source': 'pattern_comparison', 'pattern': 'sequential'},
    },
    'sequential_resolution_reviewer': {
        'name': 'PatternComparison-Sequential-ResolutionReviewer',
        'instructions': 'You are a senior banking reviewer preparing the final resolution summary. Synthesize prior analysis into a clear action plan with investigation summary, protections, provisional credit view, open questions, and next steps.',
        'metadata': {'source': 'pattern_comparison', 'pattern': 'sequential'},
    },
    'handoff_router_agent': {
        'name': 'PatternComparison-Handoff-RouterAgent',
        'instructions': 'You are an orchestration router for banking fraud operations. Choose exactly one specialist: fraud_analyst, security_agent, or credits_agent. Return exactly two lines: SPECIALIST: <name> and REASON: <brief reason>.',
        'metadata': {'source': 'pattern_comparison', 'pattern': 'handoff'},
    },
    'handoff_fraud_analyst': {
        'name': 'PatternComparison-Handoff-FraudAnalyst',
        'instructions': 'You are a bank fraud analyst handling a routed case. Explain suspicious activity, transaction-level risk, likely fraud patterns, and what should be investigated first.',
        'metadata': {'source': 'pattern_comparison', 'pattern': 'handoff'},
    },
    'handoff_security_agent': {
        'name': 'PatternComparison-Handoff-SecurityAgent',
        'instructions': 'You are a banking security specialist handling a routed case. Recommend urgent account protection actions, monitoring controls, and customer containment steps.',
        'metadata': {'source': 'pattern_comparison', 'pattern': 'handoff'},
    },
    'handoff_credits_agent': {
        'name': 'PatternComparison-Handoff-CreditsAgent',
        'instructions': 'You are a provisional credit specialist handling a routed case. Evaluate provisional credit suitability, required evidence, timing considerations, and policy caveats.',
        'metadata': {'source': 'pattern_comparison', 'pattern': 'handoff'},
    },
    'concurrent_fraud_analyst': {
        'name': 'PatternComparison-Concurrent-FraudAnalyst',
        'instructions': 'You are a bank fraud analyst. Independently analyze the case and provide a concise fraud investigation perspective.',
        'metadata': {'source': 'pattern_comparison', 'pattern': 'concurrent'},
    },
    'concurrent_security_agent': {
        'name': 'PatternComparison-Concurrent-SecurityAgent',
        'instructions': 'You are a banking security specialist. Independently provide the account protection and containment perspective for the case.',
        'metadata': {'source': 'pattern_comparison', 'pattern': 'concurrent'},
    },
    'concurrent_credits_agent': {
        'name': 'PatternComparison-Concurrent-CreditsAgent',
        'instructions': 'You are a provisional credit specialist. Independently provide the claims and provisional credit perspective for the case.',
        'metadata': {'source': 'pattern_comparison', 'pattern': 'concurrent'},
    },
    'group_chat_fraud_analyst': {
        'name': 'PatternComparison-GroupChat-FraudAnalyst',
        'instructions': 'You are a fraud analyst participating in a group discussion. Build on previous messages, add new fraud insights, and avoid repeating earlier points.',
        'metadata': {'source': 'pattern_comparison', 'pattern': 'group_chat'},
    },
    'group_chat_security_agent': {
        'name': 'PatternComparison-GroupChat-SecurityAgent',
        'instructions': 'You are a banking security specialist participating in a group discussion. Build on previous messages with containment actions and operational safeguards.',
        'metadata': {'source': 'pattern_comparison', 'pattern': 'group_chat'},
    },
    'group_chat_credits_agent': {
        'name': 'PatternComparison-GroupChat-CreditsAgent',
        'instructions': 'You are a provisional credit specialist participating in a group discussion. Build on previous messages with claims, documentation, and provisional credit guidance.',
        'metadata': {'source': 'pattern_comparison', 'pattern': 'group_chat'},
    },
    'magentic_planner': {
        'name': 'PatternComparison-Magentic-Planner',
        'instructions': 'You are a strategic planner for banking fraud cases. Decompose the case into a prioritized task list. Return ONLY a JSON array of objects with keys: task_id, description, assigned_to (one of: researcher, writer, validator), priority (1-3). Example: [{"task_id": "T1", "description": "...", "assigned_to": "researcher", "priority": 1}]',
        'metadata': {'source': 'pattern_comparison', 'pattern': 'magentic'},
    },
    'magentic_researcher': {
        'name': 'PatternComparison-Magentic-Researcher',
        'instructions': 'You are a fraud case researcher. Given a specific subtask from the task ledger, investigate thoroughly and provide detailed findings. Be concise and evidence-focused. Reference the task_id you are working on.',
        'metadata': {'source': 'pattern_comparison', 'pattern': 'magentic'},
    },
    'magentic_writer': {
        'name': 'PatternComparison-Magentic-Writer',
        'instructions': 'You are a fraud case report writer. Synthesize all research findings into a structured resolution document with sections: Executive Summary, Transaction Analysis, Risk Assessment, Recommended Actions, Provisional Credit Assessment, and Next Steps.',
        'metadata': {'source': 'pattern_comparison', 'pattern': 'magentic'},
    },
    'magentic_validator': {
        'name': 'PatternComparison-Magentic-Validator',
        'instructions': 'You are a senior quality validator for banking fraud resolutions. Review the draft report for completeness, accuracy, and regulatory compliance. If acceptable, respond with DECISION: APPROVE followed by brief summary. If revisions needed, respond with DECISION: REVISE followed by specific feedback.',
        'metadata': {'source': 'pattern_comparison', 'pattern': 'magentic'},
    },
    'group_chat_moderator': {
        'name': 'PatternComparison-GroupChat-Moderator',
        'instructions': 'You are the moderator for a banking fraud resolution discussion. Review the conversation, identify gaps, and end with DECISION: continue or DECISION: conclude.',
        'metadata': {'source': 'pattern_comparison', 'pattern': 'group_chat'},
    },
}

for key, config in AGENT_CONFIGS.items():
    print(f"{key:32} -> {config['name']}")


## 4. Create or find Foundry agents


In [ ]:
agents = {}
for key, config in AGENT_CONFIGS.items():
    agents[key] = get_or_create_agent(
        client=client,
        name=config['name'],
        instructions=config['instructions'],
        model=MODEL_DEPLOYMENT,
        metadata=config['metadata'],
    )

print('\nResolved agents:')
for key, agent in agents.items():
    print(f'- {key}: {agent.name} ({agent.id})')


## 5. Helper — Run an agent via thread + run and extract response text


In [ ]:
def extract_text_from_message(message):
    parts = []
    for item in getattr(message, 'text_messages', []) or []:
        text_details = getattr(item, 'text', None)
        value = getattr(text_details, 'value', None)
        if value:
            parts.append(value)
    return '\n'.join(parts).strip()

def history_to_thread_messages(history_entries):
    return [
        ThreadMessageOptions(
            role='user',
            content=f"[{entry['speaker']}] {entry['content']}",
        )
        for entry in history_entries
    ]

def run_agent(client, agent_id, task=None, thread_messages=None, run_metadata=None):
    if thread_messages is None:
        if task is None:
            raise ValueError('Provide either task or thread_messages')
        thread_messages = [ThreadMessageOptions(role='user', content=task)]

    run = client.create_thread_and_process_run(
        agent_id=agent_id,
        thread=AgentThreadCreationOptions(messages=thread_messages),
        metadata=run_metadata or {},
    )

    if run.status != RunStatus.COMPLETED:
        raise RuntimeError(f'Run failed with status {run.status}: {getattr(run, "last_error", None)}')

    messages = list(client.messages.list(thread_id=run.thread_id, run_id=run.id, order='asc'))
    assistant_messages = [message for message in messages if getattr(message, 'role', None) == 'assistant']
    response_text = '\n\n'.join(
        filter(None, [extract_text_from_message(message) for message in assistant_messages])
    ).strip()

    return {
        'run_id': run.id,
        'thread_id': run.thread_id,
        'status': run.status,
        'response': response_text,
        'messages': messages,
    }

def preview(text, limit=500):
    text = text or ''
    return text if len(text) <= limit else text[:limit] + '...'


## 6. Pattern 1 — Sequential


In [ ]:
start = time.perf_counter()
sequential_steps = []
sequential_history = [{'speaker': 'customer', 'content': BANKING_TASK}]

for step_name in [
    'sequential_fraud_analyst',
    'sequential_security_agent',
    'sequential_credits_agent',
    'sequential_resolution_reviewer',
]:
    result = run_agent(
        client=client,
        agent_id=agents[step_name].id,
        thread_messages=history_to_thread_messages(sequential_history),
        run_metadata={'pattern': 'sequential', 'source': 'pattern_comparison'},
    )
    sequential_steps.append({
        'agent': step_name,
        'response': result['response'],
        'thread_id': result['thread_id'],
        'run_id': result['run_id'],
    })
    sequential_history.append({'speaker': step_name, 'content': result['response']})

results['Sequential'] = {
    'duration_seconds': round(time.perf_counter() - start, 2),
    'steps': sequential_steps,
    'final_output': sequential_steps[-1]['response'],
}

print(f"Sequential completed in {results['Sequential']['duration_seconds']}s")
for step in sequential_steps:
    print(f"\n[{step['agent']}]")
    print(preview(step['response']))


## 7. Pattern 2 — Handoff


In [ ]:
start = time.perf_counter()
router_prompt = (
    f'{BANKING_TASK}\n\n'
    'Decide which specialist should take the first action. '
    'Return exactly two lines: SPECIALIST: <fraud_analyst|security_agent|credits_agent> and REASON: <reason>.'
)

router_result = run_agent(
    client=client,
    agent_id=agents['handoff_router_agent'].id,
    task=router_prompt,
    run_metadata={'pattern': 'handoff', 'source': 'pattern_comparison'},
)

router_text = router_result['response']
selected_specialist = 'handoff_fraud_analyst'
routing_map = {
    'fraud_analyst': 'handoff_fraud_analyst',
    'security_agent': 'handoff_security_agent',
    'credits_agent': 'handoff_credits_agent',
}
for token, mapped_agent in routing_map.items():
    if token in router_text.lower():
        selected_specialist = mapped_agent
        break

handoff_history = [
    {'speaker': 'customer', 'content': BANKING_TASK},
    {'speaker': 'router_agent', 'content': router_text},
]

specialist_result = run_agent(
    client=client,
    agent_id=agents[selected_specialist].id,
    thread_messages=history_to_thread_messages(handoff_history),
    run_metadata={'pattern': 'handoff', 'source': 'pattern_comparison'},
)

results['Handoff'] = {
    'duration_seconds': round(time.perf_counter() - start, 2),
    'router_output': router_text,
    'selected_specialist': selected_specialist,
    'specialist_output': specialist_result['response'],
    'final_output': specialist_result['response'],
}

print(f"Handoff completed in {results['Handoff']['duration_seconds']}s")
print('\n[handoff_router_agent]')
print(router_text)
print(f"\n[selected specialist] {selected_specialist}")
print(preview(specialist_result['response']))


## 8. Pattern 3 — Concurrent


In [ ]:
start = time.perf_counter()
concurrent_agents = [
    'concurrent_fraud_analyst',
    'concurrent_security_agent',
    'concurrent_credits_agent',
]
concurrent_results = {}

def concurrent_worker(agent_key):
    return agent_key, run_agent(
        client=client,
        agent_id=agents[agent_key].id,
        task=BANKING_TASK,
        run_metadata={'pattern': 'concurrent', 'source': 'pattern_comparison'},
    )

with ThreadPoolExecutor(max_workers=len(concurrent_agents)) as executor:
    futures = [executor.submit(concurrent_worker, key) for key in concurrent_agents]
    for future in as_completed(futures):
        key, result = future.result()
        concurrent_results[key] = result

concurrent_summary = '\n\n'.join(
    f'[{key}]\n{concurrent_results[key]["response"]}'
    for key in concurrent_agents
)

results['Concurrent'] = {
    'duration_seconds': round(time.perf_counter() - start, 2),
    'agent_outputs': {key: concurrent_results[key]['response'] for key in concurrent_agents},
    'final_output': concurrent_summary,
}

print(f"Concurrent completed in {results['Concurrent']['duration_seconds']}s")
for key in concurrent_agents:
    print(f"\n[{key}]")
    print(preview(concurrent_results[key]['response']))


## 9. Pattern 4 — Group Chat


In [ ]:
start = time.perf_counter()
group_sequence = [
    'group_chat_fraud_analyst',
    'group_chat_security_agent',
    'group_chat_credits_agent',
]
group_history = [{'speaker': 'customer', 'content': BANKING_TASK}]
group_rounds = []
max_rounds = 2

for round_number in range(1, max_rounds + 1):
    round_entries = []
    for agent_key in group_sequence:
        result = run_agent(
            client=client,
            agent_id=agents[agent_key].id,
            thread_messages=history_to_thread_messages(group_history),
            run_metadata={'pattern': 'group_chat', 'source': 'pattern_comparison'},
        )
        round_entries.append({'agent': agent_key, 'response': result['response']})
        group_history.append({'speaker': agent_key, 'content': result['response']})

    moderator_result = run_agent(
        client=client,
        agent_id=agents['group_chat_moderator'].id,
        thread_messages=history_to_thread_messages(group_history),
        run_metadata={'pattern': 'group_chat', 'source': 'pattern_comparison'},
    )
    round_entries.append({'agent': 'group_chat_moderator', 'response': moderator_result['response']})
    group_history.append({'speaker': 'group_chat_moderator', 'content': moderator_result['response']})
    group_rounds.append(round_entries)

    if 'decision: conclude' in moderator_result['response'].lower():
        break

results['Group Chat'] = {
    'duration_seconds': round(time.perf_counter() - start, 2),
    'rounds': group_rounds,
    'final_output': '\n\n'.join(
        f"[{entry['speaker']}] {entry['content']}" for entry in group_history
    ),
}

print(f"Group Chat completed in {results['Group Chat']['duration_seconds']}s")
for idx, round_entries in enumerate(group_rounds, start=1):
    print(f"\n=== Round {idx} ===")
    for entry in round_entries:
        print(f"\n[{entry['agent']}]")
        print(preview(entry['response'], limit=350))


## 10. Pattern 5 -- Magentic (Task Ledger)

**How it works:** A Planner agent decomposes the case into a prioritized task ledger.
A Researcher works through each subtask. A Writer synthesizes findings into a report.
A Validator reviews and can request revisions -- creating an iterative feedback loop.

```
Customer Request
     |
Planner   ->  decomposes into subtask ledger (JSON)
     |
Researcher  ->  investigates each subtask
     |
Writer  ->  synthesizes resolution report
     |
Validator  ->  APPROVE or REVISE
     | (if REVISE, loops back to Writer, max 2 rounds)
Final Report
```

**Key insight:** Unlike Sequential (fixed pipeline, no feedback), Magentic uses a
**task ledger** for structured decomposition and a **validator loop** for iterative
quality refinement. The planner drives the strategy; the validator enforces quality.


In [ ]:
import json as _json

start = time.perf_counter()
magentic_steps = []
task_ledger = []

# Phase 1: Planner decomposes case into subtasks
planner_result = run_agent(
    client=client,
    agent_id=agents['magentic_planner'].id,
    task=BANKING_TASK,
    run_metadata={'pattern': 'magentic', 'phase': 'planning'},
)
magentic_steps.append({'agent': 'magentic_planner', 'response': planner_result['response']})
print('[magentic_planner]')
print(preview(planner_result['response'], 400))

# Parse task ledger from planner output
try:
    import re
    json_match = re.search(r'\[.*\]', planner_result['response'], re.DOTALL)
    task_ledger = _json.loads(json_match.group()) if json_match else []
except Exception:
    task_ledger = [{'task_id': 'T1', 'description': 'Full case analysis', 'assigned_to': 'researcher', 'priority': 1}]
print(f'\nTask ledger: {len(task_ledger)} subtasks')

# Phase 2: Researcher works through each subtask
research_history = [{'speaker': 'customer', 'content': BANKING_TASK}]
research_history.append({'speaker': 'planner', 'content': planner_result['response']})

for task in sorted(task_ledger, key=lambda t: t.get('priority', 99)):
    subtask_prompt = f"SUBTASK {task.get('task_id', '?')}: {task.get('description', 'investigate')}"
    research_history.append({'speaker': 'task_ledger', 'content': subtask_prompt})

    r = run_agent(
        client=client,
        agent_id=agents['magentic_researcher'].id,
        thread_messages=history_to_thread_messages(research_history),
        run_metadata={'pattern': 'magentic', 'phase': 'research', 'task_id': task.get('task_id', '?')},
    )
    research_history.append({'speaker': 'researcher', 'content': r['response']})
    magentic_steps.append({'agent': f"researcher ({task.get('task_id', '?')})", 'response': r['response']})
    print(f"\n[researcher - {task.get('task_id', '?')}] done ({len(r['response'])} chars)")

# Phase 3: Writer synthesizes
writer_history = research_history.copy()
writer_result = run_agent(
    client=client,
    agent_id=agents['magentic_writer'].id,
    thread_messages=history_to_thread_messages(writer_history),
    run_metadata={'pattern': 'magentic', 'phase': 'writing'},
)
magentic_steps.append({'agent': 'magentic_writer', 'response': writer_result['response']})
print(f'\n[magentic_writer] done ({len(writer_result["response"])} chars)')

# Phase 4: Validator loop (max 2 revision rounds)
draft = writer_result['response']
max_revisions = 2
revision_count = 0

for rev_round in range(max_revisions + 1):
    val_history = research_history + [
        {'speaker': 'writer', 'content': draft},
    ]
    val_result = run_agent(
        client=client,
        agent_id=agents['magentic_validator'].id,
        thread_messages=history_to_thread_messages(val_history),
        run_metadata={'pattern': 'magentic', 'phase': 'validation', 'round': rev_round},
    )
    magentic_steps.append({'agent': f'validator (round {rev_round})', 'response': val_result['response']})
    print(f"\n[validator round {rev_round}] {val_result['response'][:100]}")

    if 'APPROVE' in val_result['response'].upper():
        print('Validator approved!')
        break

    if rev_round < max_revisions:
        revision_count += 1
        revise_history = research_history + [
            {'speaker': 'writer', 'content': draft},
            {'speaker': 'validator', 'content': val_result['response']},
        ]
        revised = run_agent(
            client=client,
            agent_id=agents['magentic_writer'].id,
            thread_messages=history_to_thread_messages(revise_history),
            run_metadata={'pattern': 'magentic', 'phase': 'revision', 'round': rev_round},
        )
        draft = revised['response']
        magentic_steps.append({'agent': f'writer (revision {revision_count})', 'response': draft})
        print(f'[writer revision {revision_count}] done ({len(draft)} chars)')

results['Magentic'] = {
    'duration_seconds': round(time.perf_counter() - start, 2),
    'steps': magentic_steps,
    'task_ledger': task_ledger,
    'revision_count': revision_count,
    'final_output': draft,
}

print(f"\nMagentic completed in {results['Magentic']['duration_seconds']}s")
print(f'Task ledger: {len(task_ledger)} subtasks | Revisions: {revision_count}')
print('\n--- Final Report Preview ---')
print(preview(draft, 600))


## 11. Comparison

Timing and summary table for all five orchestration patterns.


In [ ]:
comparison = [
    ('Sequential', 'Agents run in series, each builds on previous', 'Multi-step workflows with dependencies'),
    ('Handoff', 'Router picks one specialist', 'Routing to domain experts, call centers'),
    ('Concurrent', 'All agents run in parallel', 'Independent analyses needed fast'),
    ('Group Chat', 'Agents discuss in rounds', 'Complex decisions requiring debate'),
    ('Magentic', 'Planner, task ledger, writer, validator loop', 'Complex cases needing structured decomposition and quality gates'),
]

def summary_count(pattern_name, payload):
    if pattern_name == 'Sequential':
        return len(payload.get('steps', []))
    if pattern_name == 'Handoff':
        return 2 if payload else 0
    if pattern_name == 'Concurrent':
        return len(payload.get('agent_outputs', {}))
    if pattern_name == 'Group Chat':
        return sum(len(round_entries) for round_entries in payload.get('rounds', []))
    if pattern_name == 'Magentic':
        return len(payload.get('steps', []))
    return 0

print(f"{'Pattern':<12} {'Time (s)':>10} {'Steps':>8}  {'How It Works'}")
print(f"{'-' * 12} {'-' * 10} {'-' * 8}  {'-' * 55}")
for name, how, best_for in comparison:
    payload = results.get(name)
    if payload:
        duration = payload.get('duration_seconds', '--')
        steps = summary_count(name, payload)
        print(f"{name:<12} {duration:>10} {steps:>8}  {how}")
    else:
        print(f"{name:<12} {'--':>10} {'--':>8}  {how}")

print('\nBest fit by pattern:')
for name, how, best_for in comparison:
    print(f'- {name}: {best_for}')


## 12. Cleanup

Delete the persistent Foundry agents created for this notebook when you no longer need them.
Set `DELETE_PATTERN_COMPARISON_AGENTS=1` in `.env` to actually remove them.


In [ ]:
delete_agents = os.getenv('DELETE_PATTERN_COMPARISON_AGENTS', '0') == '1'

if not delete_agents:
    print('Skipping cleanup. Set DELETE_PATTERN_COMPARISON_AGENTS=1 in .env to delete the created agents.')
    print('Tracked agents:')
    for key, agent in agents.items():
        print(f"- {key}: {getattr(agent, 'name', key)} ({agent.id})")
else:
    for key, agent in agents.items():
        client.delete_agent(agent.id)
        print(f"Deleted {key}: {agent.id}")
